# 13. Learned compression with independently simulated bin yields

Repeat the principal comparison of notebook 08 using the **trained
density-ratio networks to define the observable**. The likelihood in
both binned constructions comes from physical simulator yields. Compare
analytical unbinned inference, an analytical ratio map with 36 bins,
and a learned ratio map with the same 36 bins. The numerator endpoint
floor is 0.2 and the transformation power is 100, as in 08–11.

Two complementary Asimov comparisons separate compression from
finite-simulation closure: matched bank A data/templates, and independent
bank B data with bank A templates. No network is retrained, no learned
density weights define physical yields, and no spline approximation is
used. This notebook does not measure repeated-experiment coverage.

### Reuse the trained v4 models

Use the same **`RUN_NAME = "paper-distinct-s-v4"`**, `MODE`, and
`CONFIG_OVERRIDES` as your completed notebooks 01–03. Run **13 only** after
updating the source and restarting the runtime; no retraining or rerunning
08–12 is needed. The saved preselection and ratio networks are frozen.

The bootstrap below reuses an existing `/content/poodemo` checkout. In Colab,
delete that temporary checkout and restart if it contains older source; the
saved Drive run is separate. Private-repository access follows the earlier
notebooks: a `GITHUB_TOKEN` Colab secret with Contents: read, the hidden prompt,
or `/content/poodemo_source.zip`. `JAX_BACKEND="auto"` uses an available GPU.

This notebook generates **two new, independent simulation integration banks**
with deterministic, separate random streams. Each bank has two million
generated events per S, B, and NI proposal stratum in production, before the
existing preselection; smoke mode uses 20,000 per stratum. These are integration
events, not new training samples and not pseudo-experiments. The physical
expected yields and experimental exposure remain those of the saved run.
The two banks require time and disk space; progress is printed and their
verified cache is reused when settings and source artifacts match.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "paper-distinct-s-v4"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}
JAX_BACKEND = os.environ.get("POODEMO_JAX_BACKEND", "auto")  # auto, gpu, or cpu

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    requested_backend = JAX_BACKEND.strip().lower()
    if requested_backend not in ("auto", "gpu", "cpu"):
        raise ValueError("JAX_BACKEND must be auto, gpu, or cpu")
    selected_backend = requested_backend
    if requested_backend == "auto":
        try:
            gpu_probe = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                       capture_output=True, text=True, timeout=10)
            gpu_present = gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip())
        except (OSError, subprocess.TimeoutExpired):
            gpu_present = False
        selected_backend = "gpu" if gpu_present else "cpu"
    # CUDA-only selection fails visibly if initialization fails. Keep PyTorch's
    # CUDA visibility and avoid JAX reserving most GPU memory before training.
    jax_platform = "cuda" if selected_backend == "gpu" else "cpu"
    os.environ["JAX_PLATFORMS"] = jax_platform
    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    REPO_DIR = Path("/content/poodemo")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import tempfile
        import urllib.error
        import urllib.request
        import zipfile

        uploaded_zip = Path("/content/poodemo_source.zip")
        if uploaded_zip.exists():
            archive_bytes = uploaded_zip.read_bytes()
            archive_kind = "zip"
        else:
            archive_url = "https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/main"

            def fetch_source(token=None):
                headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
                if token:
                    headers["Authorization"] = "Bearer " + token
                request = urllib.request.Request(archive_url, headers=headers)
                with urllib.request.urlopen(request, timeout=180) as response:
                    return response.read()

            try:
                archive_bytes = fetch_source()
            except urllib.error.HTTPError as error:
                if error.code not in (401, 403, 404):
                    raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
                token = None
                try:
                    from google.colab import userdata
                    token = userdata.get("GITHUB_TOKEN")
                except Exception:
                    pass
                if not token:
                    token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
                if not token:
                    raise RuntimeError("Upload /content/poodemo_source.zip or provide read access.")
                try:
                    archive_bytes = fetch_source(token)
                except urllib.error.HTTPError as auth_error:
                    raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
                finally:
                    token = None
            archive_kind = "tar"

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            if archive_kind == "zip":
                with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                    for entry in archive.infolist():
                        safe_destination(entry.filename)
                        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                            raise RuntimeError("Symbolic links are not supported in the source ZIP.")
                    archive.extractall(staging)
            else:
                with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                    for entry in archive.getmembers():
                        safe_destination(entry.name)
                        if not (entry.isfile() or entry.isdir()):
                            raise RuntimeError("Links and special files are not supported in the source archive.")
                    archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The uploaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        from importlib.metadata import version, PackageNotFoundError
        # JAX discovers every installed plugin, even when selecting CPU.
        # Retain already-matched CUDA 12 packages when rerunning GPU setup.
        remove_plugins = []
        for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt", "jax-cuda13-plugin", "jax-cuda13-pjrt"):
            try:
                installed_version = version(package)
            except PackageNotFoundError:
                continue
            if not (selected_backend == "gpu" and package.startswith("jax-cuda12-") and installed_version == "0.5.3"):
                remove_plugins.append(package)
        if remove_plugins:
            subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *remove_plugins])
        gpu_requirements = (["jax==0.5.3", "jaxlib==0.5.3",
                             "jax-cuda12-plugin[with-cuda]==0.5.3", "jax-cuda12-pjrt==0.5.3"]
                            if selected_backend == "gpu" else [])
        # Resolve with the base requirements so PyTorch's CUDA dependencies are
        # considered together with JAX's. Avoid blanket --upgrade changes.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                               str(REPO_DIR / "requirements-colab.txt"), *gpu_requirements])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

        # Check the live notebook kernel before any expensive network training.
        import jax
        import jaxlib
        import jax.numpy as jnp
        if jax.__version__ != "0.5.3" or jaxlib.__version__ != "0.5.3":
            raise RuntimeError("JAX packages are stale in this kernel. Restart the runtime and rerun setup.")
        if selected_backend == "gpu":
            for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt"):
                if version(package) != "0.5.3":
                    raise RuntimeError(f"{package} must match JAX 0.5.3. Restart the runtime and rerun setup.")
        jax.config.update("jax_platforms", jax_platform)
        jax.config.update("jax_enable_x64", True)
        try:
            jax_devices = jax.devices()
            if not jax_devices or any(device.platform != selected_backend for device in jax_devices):
                raise RuntimeError(f"Expected {selected_backend} devices; found {jax_devices}")
            probe_value, probe_gradient = jax.jit(jax.value_and_grad(lambda x: jnp.sum(x*x)))(
                jnp.asarray([1., 2., 3.], dtype=jnp.float64))
            probe_gradient.block_until_ready()
            if (float(probe_value) != 14. or list(probe_gradient) != [2., 4., 6.]
                    or probe_gradient.dtype != jnp.float64
                    or any(device.platform != selected_backend for device in probe_gradient.devices())):
                raise RuntimeError("JAX double-precision value/gradient check failed")
        except Exception as error:
            raise RuntimeError(
                f"JAX {selected_backend} startup failed. Select a Colab GPU runtime if requesting GPU, "
                "then restart the runtime and rerun setup. Use JAX_BACKEND='cpu' only for an intentional CPU run."
            ) from error
        print(f"JAX {jax.__version__} / jaxlib {jaxlib.__version__}: backend={selected_backend}, {jax_devices}; float64 JIT/gradient OK")

    # A new editable install is not activated in an already-running kernel.
    # Expose the inner package directly, including after a failed import cached
    # the outer /content/poodemo checkout as a namespace package.
    import importlib
    source_root = str(REPO_DIR.resolve())
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)
    importlib.invalidate_caches()
    cached_package = sys.modules.get("poodemo")
    cached_file = getattr(cached_package, "__file__", None)
    expected_init = REPO_DIR / "poodemo" / "__init__.py"
    if cached_package is not None and (cached_file is None or Path(cached_file).resolve() != expected_init.resolve()):
        for module_name in list(sys.modules):
            if module_name == "poodemo" or module_name.startswith("poodemo."):
                del sys.modules[module_name]
else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))


import mplhep as hep
hep.style.use("ATLAS")
# Typography/ticks only; no ATLAS label.
plt.rcParams.update({"axes.grid": False, "figure.dpi": 110})

## 1. Freeze the observable separately from the physical likelihood

The two observable choices are
\[
t_\eta^{\rm ana}(x)=\operatorname{sigmoid}
\left[a\log\frac{r(x;\eta_{\rm eff})}{r(x;1)}\right],\qquad
t_\eta^{\rm NN}(x)=\operatorname{sigmoid}
\left[a\log\frac{\widehat r(x;\eta_{\rm eff})}{\widehat r(x;1)}\right],
\quad \eta_{\rm eff}=\max(\eta,0.2).
\]
The saved process-ratio networks and their original integration-bank
normalizers are frozen before evaluating either new bank. Analytical
and learned map normalizations use the same fixed selected physical
rates. We do not recalibrate either map on bank B or tune it for closure.

At each tested physical \(\mu_0\), choose \(\eta=\mu_0\), fill the
observed histogram, and freeze its partition for **both** the constrained
and free fits. The denominator fit is repeated for each partition.
Only the observable numerator is floored; the physical \(\mu\), yield
coefficients, and free-fit domain retain their original definitions.
All hypotheses below the floor share a partition, while their predicted
yields remain different. At \(\eta=1\) both maps collapse to 0.5,
contained inside a bin; event-count information remains.

Change `MU_GRID` for the physical scan or the plotting limits below for
display. Changing the simulation-bank settings or frozen input artifacts
requires a distinct `OUTPUT_TAG`; prior results and earlier notebooks
remain separate. The defaults preserve the completed v4 physics run.

In [ ]:
from poodemo.learned_compression_banks import prepare_compression_banks
from poodemo.learned_compression_study import run_learned_compression_study
from poodemo.learned_compression_plots import (
    plot_compression_scans, plot_compression_information,
    plot_compression_distributions, plot_compression_closure,
)

N_BINS = 36
REFERENCE_RATIO_A = 100.0
REFERENCE_RATIO_ETA_MIN = 0.2
N_PER_SOURCE = 20_000 if MODE == "smoke" else 2_000_000
SEED = 131013
CHUNK_SIZE = 100_000
OUTPUT_TAG = "13_learned_compression_36_floor02"
SYSTEMATICS = (False, True)
MU_GRID = None  # None uses the saved physical scan grid; no physical floor.
DISTRIBUTION_ETAS = (0.0, 0.3, 0.6, 1.0, 1.5)

# Display limits only: the complete scan is calculated and saved.
SCAN_XLIM = (0.02, 1.15)
SCAN_YLIM = (0.0, 8.0)
INFORMATION_XLIM = None

FIGURE_DIR = ROOT / "plots" / OUTPUT_TAG
print(f"{N_BINS} bins; a={REFERENCE_RATIO_A:g}; numerator eta_min={REFERENCE_RATIO_ETA_MIN:g}")
print(f"Two banks × three strata × {N_PER_SOURCE:,} events before preselection")
print("Physical Asimov truth:", run.config["asimov_mu"])
print("Physical free-fit bounds:", run.config["mu_fit_bounds"])
print("Results:", ROOT / "results" / OUTPUT_TAG)

## 2. Independent physical simulation banks

Generate stratified S/B/NI proposals with independent deterministic
streams for A and B, then apply the saved preselection. Evaluate the
analytical component intensities and NI variations on the surviving
events. The proposal/integration weights account for generated events
that fail selection; S, SBI, B, and NI template integrals are estimates
of absolute selected physical yields.

The trained networks are evaluated on these same selected events **only
to assign bins**. Negative or nonfinite combined learned intensities
invalidate the corresponding ratio map and are reported explicitly;
they are not hidden by dropping events or by treating negative density
estimates as physical yields. Bank caches include source fingerprints
and payload checksums.

In [ ]:
banks = prepare_compression_banks(
    run, n_per_source=N_PER_SOURCE, seed=SEED,
    chunk_size=CHUNK_SIZE, output_tag=OUTPUT_TAG,
)

bank_rows = []
for bank_name in ("template", "validation"):
    meta = banks[bank_name]["metadata"]
    for source in ("S", "B", "NI"):
        bank_rows.append({
            "bank": bank_name, "source": source,
            "generated": meta["generated_counts"][source],
            "selected": meta["selected_counts"][source],
            "stream_id": meta["stream_id"],
        })
display(pd.DataFrame(bank_rows))
print("Frozen rates from the original saved integration bank:")
display(pd.Series(banks["rates"], index=("S", "SBI", "B", "NI"), name="selected_rate"))
print("Frozen NN normalizers from the original saved integration bank:")
display(pd.Series(banks["metadata"]["frozen_normalizers"]))
print("Source and bank fingerprint:", banks["metadata"]["fingerprint"])
print("Cache:", banks["directory"])

## 3. Construct direct templates and fit both Asimov comparisons

For map \(t_\eta\), the bin prediction is
\[
\Lambda_b(\mu,\alpha;\eta)=
\int\mathbf{1}[t_\eta(x)\in B_b]\,
\lambda(x;\mu,\alpha)\,dx.
\]
The simulation estimate of this integral uses physical component
intensities. A neural approximation changes the partition, not the
intensity used to predict its yield.

| Comparison | Asimov event integration | Binned template integration | Purpose |
|---|---|---|---|
| Matched | Bank A | Bank A | Compression/binning comparison with a common numerical expectation |
| Independent | Bank B | Bank A | Detect sensitivity to independent finite simulation |

Both comparisons use the same physical Asimov truth and independently
fitted analytical unbinned baseline. The baseline is recomputed for these
banks rather than imported from notebook 03. The matched construction
shares numerical fluctuations and therefore cannot establish independent
closure; the second comparison addresses that limitation.

NI fixed and NI profiled are both calculated. As in 09, the profiled
binned model morphs nominal/down/up NI yields **after integration**.
The analytical unbinned model morphs event-level intensities. Their
difference can include nuisance-interpolation effects in addition to
compression. The NI-fixed comparison isolates compression more directly.

In [ ]:
study = run_learned_compression_study(
    run, banks, n_bins=N_BINS, reference_power=REFERENCE_RATIO_A,
    reference_eta_floor=REFERENCE_RATIO_ETA_MIN,
    scan_grid=MU_GRID, systematics=SYSTEMATICS,
    distribution_etas=DISTRIBUTION_ETAS, output_tag=OUTPUT_TAG,
)

display(pd.Series(study["metadata"], name="study metadata"))
diagnostics = study["map_diagnostics"]
print("Map validity summary (one row per bank, map, and construction point):")
display(diagnostics.groupby(["bank", "model"], dropna=False).agg(
    construction_points=("valid", "size"),
    valid_points=("valid", "sum"),
    maximum_invalid_fraction=("invalid_fraction", "max"),
    maximum_invalid_expected_fraction=("invalid_expected_fraction", "max"),
))
invalid_maps = diagnostics.loc[~diagnostics.valid.fillna(False)]
if len(invalid_maps):
    print("Invalid learned intensities prevent a valid log-ratio map at these points; inspect before interpreting:")
    display(invalid_maps)
invalid_fits = study["scans"].loc[~study["scans"].valid.fillna(False)]
if len(invalid_fits):
    print("Unsuccessful or unavailable fits are retained in the tables and omitted from plotted lines:")
    display(invalid_fits)

In [ ]:
def show_figures(figures):
    for name, figure in figures.items():
        print(name)
        display(figure)
        plt.close(figure)

## 4. Likelihood scans and differences from analytical unbinned

Compare the two binned curves against the analytical unbinned curve,
and against each other. The residual panel shows
\(\Delta q_\mu=q_\mu^{\rm binned}-q_\mu^{\rm unbinned}\) for the
**same** Asimov comparison and nuisance treatment. Inspect matched and
independent results separately. A binned curve need not differ from the
unbinned curve with one sign at every point of a profiled scan.

These plots keep the familiar zoom from 08. Complete scan tables and
the full-range plots below retain every calculated point. Failed fits
and invalid maps remain visible in the diagnostic tables; their plotted
values are gaps, not zero test statistics.

In [ ]:
show_figures(plot_compression_scans(
    study, xlim=SCAN_XLIM, ylim=SCAN_YLIM, output=FIGURE_DIR / "zoom"))
show_figures(plot_compression_scans(
    study, output=FIGURE_DIR / "full_range"))

## 5. Fitted values and local information

Every scan point has its own frozen compression and therefore its own
unrestricted fit. The fitted \(\widehat\mu\) values below are not
interchangeable with the minimum of the assembled moving-observable
scan. Compare matched and independent fits at the same construction
point to assess finite-bank sensitivity.

The local-width calculation differentiates the **physical** model at the
Asimov truth with each map fixed. Thus the horizontal axis below labels
the map, not the physical point at which the information is evaluated.
These widths describe the nominal bank-A model; they do not replace
the independent-bank closure test or a repeated-experiment study.

In [ ]:
fit_columns = [c for c in (
    "comparison", "systematics", "model", "mu", "eta_effective",
    "mu_hat", "q", "delta_q", "valid", "fit_message",
) if c in study["scans"]]
display(study["scans"][fit_columns])
width_columns = [c for c in (
    "systematics", "model", "eta", "eta_effective", "information_mu",
    "sigma_mu", "information_fraction", "width_valid", "width_message",
) if c in study["information"]]
display(study["information"][width_columns])
show_figures(plot_compression_information(
    study, xlim=INFORMATION_XLIM, output=FIGURE_DIR / "information"))

## 6. Physical component shapes on the two maps

Compare S, SBI, B, and NI on the analytical and learned axes for
\(\eta=0,0.3,0.6,1,1.5\). The plotted component histograms are
normalized separately **for display only**; the fits retain absolute
yields. All histograms use the same 36 edges as the likelihood, including
the convention that places 0.5 inside a bin. Logarithmic axes expose tails.

In [ ]:
show_figures(plot_compression_distributions(
    study, etas=DISTRIBUTION_ETAS, output=FIGURE_DIR / "distributions"))

## 7. Independent bin-yield closure

For the physical Asimov truth, compare bank A template expectations with
bank B expectations in the identical frozen partition. There is **no
overall or binwise rescaling to enforce closure**. The lower panels show
B/A; error bars quantify independent integration-bank fluctuations,
not experimental Poisson fluctuations. The ratio errors propagate
numerator and denominator Monte Carlo uncertainties. Shared events can
correlate different bins/partitions, so these panels are diagnostics,
not a collection of independent significance tests.

Empty A bins have an undefined ratio and are retained in the tables.
Inspect effective sample sizes and pulls as well as visually smooth
curves. Increasing generated simulation, rather than the number of toys,
addresses this source of numerical uncertainty.

In [ ]:
show_figures(plot_compression_closure(
    study, etas=DISTRIBUTION_ETAS, output=FIGURE_DIR / "closure"))
closure = study["bin_closure"]
display(closure.loc[np.isclose(closure.eta, DISTRIBUTION_ETAS[0])])
print("All numerical results:", ROOT / "results" / OUTPUT_TAG)
print("All figures (PDF and PNG):", FIGURE_DIR)

## Interpretation

The matched comparison measures how the actual trained map changes the
usefulness of a 36-bin partition, on top of analytical-map binning loss.
The independent comparison tests whether those conclusions persist when
the Asimov expectation does not share the template bank's fluctuations.
These are numerical Asimov expectations integrated with physical
simulator weights, not exact closed-form integrals.

A learned map can be imperfect while its simulator-derived bin means
define a valid binned model. The central question here is the sensitivity
retained by that particular map and bin budget. Agreement in these tests
does not establish coverage, validate a learned unbinned density, or
eliminate finite-simulation uncertainties. Notebook 11 addresses the
repeated-experiment question separately; its current toys use its own
configured construction and are not rerun or overwritten here.

If the learned and analytical partitions differ appreciably, inspect the
map diagnostics, component tails, per-bin integration precision, and the
NI-fixed/profiling comparison before attributing everything to network
calibration. The saved outputs and frozen provenance allow a larger
independent-bank run to be compared without changing the trained model.